# Task 1 ablation: which of the new pieces actually help?

The new `task1.py` changes five things at once: tuned GB parameters, 5-fold fitting of the
combiner (`n_folds`), `TopKItemKNN` in place of the Jaccard `ItemKNN`, `UserKNN`, and
`RatingProfile`. This notebook adds them one at a time on the outer 10% holdout, then
drops each new source from the full model, then checks the best config against the shipped
one over several outer splits.

Reference: the shipped stack (Jaccard + adjusted + raw kNN + stats) scored **0.5454** on the
outer holdout. A0 should reproduce that; if it doesn't, the comparison isn't like-for-like.

In [3]:
import inspect
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

from assignment_1.modules.models import (
    BiasModel,
    ItemKNN,
    RatingProfile,
    RatingStats,
    StackedRating,
    TopKItemKNN,
    UserKNN,
    adjusted_cosine,
    jaccard,
)
from assignment_1.modules.tasks.task1 import TASK1_BIAS_PARAMS, TASK1_GB_PARAMS, TASK1_TRAIN_FILE
from assignment_1.modules.util import drop_zero_ratings, load_csv
from pathlib import Path

# repo root = first parent that has a data/ folder (works from notebooks/ or the root)
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir())
DATA_DIR = ROOT / "data"

OUTER_FRAC = 0.10
SEEDS = (0, 1, 2, 3)

# How the combiner is fit. OLD_FIT should give the shipped behaviour (one internal 75/25
# split). Check the signature printed below: if n_folds now defaults to 5, set
# OLD_FIT = {"n_folds": <whatever means a single split>}.
OLD_FIT: dict = {}
NEW_FIT = {"n_folds": 5, "n_jobs": 3}

# Shipped GB was sklearn defaults. random_state pins the internal early-stopping split.
DEFAULT_GB = {"random_state": 0}

train_all = drop_zero_ratings(load_csv(TASK1_TRAIN_FILE, DATA_DIR))
print(inspect.signature(StackedRating))
print(f"{len(train_all):,} nonzero training rows")

(base: assignment_1.modules.models.rating.base.RatingModel, sources: list[assignment_1.modules.models.rating.stacking.FeatureSource], regressor, val_frac: float = 0.25, seed: int = 0, scale: tuple[float, float] = (1, 5), n_folds: int | None = None, n_jobs: int = 1) -> None
1,834,017 nonzero training rows


## Sanity check: duplicate (user, item) rows
`TopKItemKNN` and `UserKNN` build sparse matrices, and `csr_matrix` **sums** duplicate
entries. Duplicates would corrupt the row pointers in `TopKItemKNN` and add residuals together
in `UserKNN`. This should print 0.

In [4]:
n_dup = int(train_all.duplicated(["user", "item"]).sum())
print(f"duplicate (user, item) rows: {n_dup}")

duplicate (user, item) rows: 0


## Configs

In [5]:
def make_sources(topk: bool = False, user: bool = False, profile: bool = False) -> list:
    """The shipped sources, with the new pieces switched on by flag."""
    jac = (TopKItemKNN(similarity=jaccard, prefix="knn_jac") if topk
           else ItemKNN(similarity=jaccard, prefix="knn_jac"))
    sources = [
        jac,
        ItemKNN(similarity=adjusted_cosine, prefix="knn_adj"),
        ItemKNN(similarity=jaccard, target="rating", prefix="knn_raw"),
    ]
    if user:
        sources.append(UserKNN())
    sources.append(RatingStats())
    if profile:
        sources.append(RatingProfile())
    return sources


CONFIGS = {
    # cumulative: add one change at a time
    "A0 shipped":            dict(gb=DEFAULT_GB,      fit=OLD_FIT, src={}),
    "A1 + tuned GB":         dict(gb=TASK1_GB_PARAMS, fit=OLD_FIT, src={}),
    "A2 + 5 folds":          dict(gb=TASK1_GB_PARAMS, fit=NEW_FIT, src={}),
    "A3 + top-k item kNN":   dict(gb=TASK1_GB_PARAMS, fit=NEW_FIT, src=dict(topk=True)),
    "A4 + user kNN":         dict(gb=TASK1_GB_PARAMS, fit=NEW_FIT, src=dict(topk=True, user=True)),
    "A5 + profile (new)":    dict(gb=TASK1_GB_PARAMS, fit=NEW_FIT,
                                  src=dict(topk=True, user=True, profile=True)),
    # leave one out from the full model (A5 − profile is A4)
    "A5 − top-k":            dict(gb=TASK1_GB_PARAMS, fit=NEW_FIT, src=dict(user=True, profile=True)),
    "A5 − user kNN":         dict(gb=TASK1_GB_PARAMS, fit=NEW_FIT, src=dict(topk=True, profile=True)),
}


def build(cfg: dict) -> StackedRating:
    return StackedRating(
        base=BiasModel(**TASK1_BIAS_PARAMS),
        sources=make_sources(**cfg["src"]),
        regressor=HistGradientBoostingRegressor(**cfg["gb"]),
        **cfg["fit"],
    )


def outer_split(df: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    hold = df.sample(frac=OUTER_FRAC, random_state=seed)
    return df.drop(hold.index), hold


def evaluate(cfg: dict, seed: int = 0) -> tuple[float, float]:
    """(holdout MSE, seconds) for one config on one outer split."""
    tr, ho = outer_split(train_all, seed)
    t0 = time.time()
    pred = build(cfg).fit(tr).predict(ho)
    mse = float(np.mean((pred - ho["rating"].to_numpy(dtype=float)) ** 2))
    return mse, time.time() - t0

## Ablation on one outer split
Slow: `UserKNN` and 5 folds multiply fit time. Each row prints as it finishes.

In [7]:
results = []
for name, cfg in CONFIGS.items():
    mse, secs = evaluate(cfg, seed=0)
    results.append({"config": name, "mse": mse, "minutes": secs / 60})
    print(f"{name:24s} {mse:.4f}  ({secs / 60:.1f} min)")

table = pd.DataFrame(results).set_index("config")
base = table.loc["A0 shipped", "mse"]
table["vs A0"] = (table["mse"] / base - 1).map(lambda x: f"{x:+.2%}")
table["step"] = table["mse"].diff().map(lambda x: f"{x:+.4f}" if pd.notna(x) else "")
table

A0 shipped               0.5430  (4.5 min)
A1 + tuned GB            0.5429  (4.7 min)
A2 + 5 folds             0.5414  (8.1 min)
A3 + top-k item kNN      0.5352  (11.7 min)
A4 + user kNN            0.5332  (12.4 min)
A5 + profile (new)       0.5299  (13.2 min)


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGKILL(-9)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

**Reading it.** For the cumulative rows (A1–A5), "step" is what that one change bought.
For the leave-one-out rows, compare against A5: if dropping a source costs less than about
0.001, that's within noise and the source isn't earning its keep. Set `BEST` to the smallest
config whose gains are all real, then check it over the other outer splits.

## Seed check: best vs shipped over several outer splits
Same test as for Task 2: a real gain should show up on every split, not just on seed 0.

In [ ]:
BEST = "A5 + profile (new)"   # change after reading the table above

rows = []
for seed in SEEDS:
    a0, _ = evaluate(CONFIGS["A0 shipped"], seed)
    best, _ = evaluate(CONFIGS[BEST], seed)
    rows.append({"seed": seed, "A0": a0, BEST: best, "diff": best - a0})
    print(f"seed {seed}: A0 {a0:.4f}  {BEST} {best:.4f}  diff {best - a0:+.4f}")

seeds = pd.DataFrame(rows).set_index("seed")
print(f"\nmean diff {seeds['diff'].mean():+.4f}, "
      f"better on {(seeds['diff'] < 0).sum()}/{len(seeds)} splits")
seeds